In [1]:
import pandas as pd
from pathlib import Path

project_folder = Path.cwd()

if project_folder.name == "notebooks":
    project_folder = project_folder.parent

data_folder = project_folder / "data"
feature_file = data_folder / "propertylens_featured.csv"

df = pd.read_csv(feature_file)


# Overall market
total_listings = len(df)

median_price = df["price_lakh"].median()
median_area = df["area_sqft"].median()
median_rate = df["rate_per_sqft"].median()

price_p90 = df["price_lakh"].quantile(0.90)
price_p99 = df["price_lakh"].quantile(0.99)


# Price segment distribution
price_segment_summary = (
    df["price_band"]
    .value_counts()
    .reindex([
        "Below 50L",
        "50L-1Cr",
        "1Cr-2Cr",
        "2Cr-5Cr",
        "5Cr-10Cr",
        "Above 10Cr"
    ])
    .fillna(0)
    .astype(int)
)

price_segment_share = (
    price_segment_summary / total_listings * 100
).round(2)


# Dominant locality
locality_counts = df["locality"].value_counts()

top_locality = locality_counts.index[0]
top_locality_count = locality_counts.iloc[0]


# Highest-priced locality with a reasonable sample
locality_price = (
    df.groupby("locality", observed=True)
    .agg(
        listings=("price_lakh", "size"),
        median_price_lakh=("price_lakh", "median"),
        median_rate_per_sqft=("rate_per_sqft", "median")
    )
)

locality_price = locality_price[
    locality_price["listings"] >= 20
]

highest_price_locality = (
    locality_price["median_price_lakh"]
    .idxmax()
)

highest_price_locality_value = (
    locality_price.loc[
        highest_price_locality,
        "median_price_lakh"
    ]
)


# Highest-rate locality with a reasonable sample
highest_rate_locality = (
    locality_price["median_rate_per_sqft"]
    .idxmax()
)

highest_rate_value = (
    locality_price.loc[
        highest_rate_locality,
        "median_rate_per_sqft"
    ]
)


# Dominant BHK
bhk_counts = (
    df.loc[df["bhk_count"] > 0, "bhk_group"]
    .value_counts()
)

dominant_bhk = bhk_counts.index[0]
dominant_bhk_count = bhk_counts.iloc[0]


# Property type
property_type_counts = df["flat_type"].value_counts()

dominant_property_type = property_type_counts.index[0]
dominant_property_type_count = property_type_counts.iloc[0]


# Segment medians
segment_summary = (
    df.groupby("price_band", observed=True)
    .agg(
        listings=("price_lakh", "size"),
        median_price_lakh=("price_lakh", "median"),
        median_area_sqft=("area_sqft", "median"),
        median_rate_per_sqft=("rate_per_sqft", "median")
    )
    .reindex([
        "Below 50L",
        "50L-1Cr",
        "1Cr-2Cr",
        "2Cr-5Cr",
        "5Cr-10Cr",
        "Above 10Cr"
    ])
)


# BHK comparison
bhk_summary = (
    df[
        (df["bhk_count"] > 0) &
        (df["bhk_count"] <= 5)
    ]
    .groupby("bhk_count", observed=True)
    .agg(
        listings=("price_lakh", "size"),
        median_price_lakh=("price_lakh", "median"),
        median_area_sqft=("area_sqft", "median"),
        median_rate_per_sqft=("rate_per_sqft", "median")
    )
    .sort_index()
)


# RERA comparison
rera_summary = (
    df.groupby("rera_approval", observed=True)
    .agg(
        listings=("price_lakh", "size"),
        median_price_lakh=("price_lakh", "median"),
        median_rate_per_sqft=("rate_per_sqft", "median"),
        median_area_sqft=("area_sqft", "median")
    )
)


# Status comparison
status_summary = (
    df[
        df["status"].isin([
            "Ready to move",
            "Under Construction"
        ])
    ]
    .groupby("status", observed=True)
    .agg(
        listings=("price_lakh", "size"),
        median_price_lakh=("price_lakh", "median"),
        median_rate_per_sqft=("rate_per_sqft", "median"),
        median_area_sqft=("area_sqft", "median")
    )
)


# Value-position summary
benchmark_counts = (
    df.groupby(
        ["locality", "flat_type"],
        observed=True
    )
    .size()
    .reset_index(name="benchmark_listings")
)

benchmarks = (
    df.groupby(
        ["locality", "flat_type"],
        observed=True
    )["rate_per_sqft"]
    .median()
    .reset_index(name="local_median_rate_per_sqft")
)

benchmarks = benchmarks.merge(
    benchmark_counts,
    on=["locality", "flat_type"],
    how="left"
)

benchmarks = benchmarks[
    benchmarks["benchmark_listings"] >= 20
]

value_df = df.merge(
    benchmarks,
    on=["locality", "flat_type"],
    how="left"
)

value_df["rate_difference_pct"] = (
    (value_df["rate_per_sqft"] -
     value_df["local_median_rate_per_sqft"])
    / value_df["local_median_rate_per_sqft"]
    * 100
)

value_df["value_position"] = pd.NA

valid = value_df["local_median_rate_per_sqft"].notna()

value_df.loc[
    valid & (value_df["rate_difference_pct"] < -10),
    "value_position"
] = "Lower-Priced"

value_df.loc[
    valid
    & (value_df["rate_difference_pct"] >= -10)
    & (value_df["rate_difference_pct"] <= 10),
    "value_position"
] = "Market-Aligned"

value_df.loc[
    valid & (value_df["rate_difference_pct"] > 10),
    "value_position"
] = "Premium-Priced"

value_position_summary = (
    value_df["value_position"]
    .value_counts()
)

benchmarked_count = value_df["value_position"].notna().sum()


# Main findings
print("PropertyLens - Final Market Findings")

print("\nMarket overview:")
print(f"Total listings analysed: {total_listings:,}")
print(f"Median property price: ₹{median_price:.2f} lakh")
print(f"Median property area: {median_area:,.0f} sqft")
print(f"Median rate: ₹{median_rate:,.0f}/sqft")
print(f"90th percentile price: ₹{price_p90:.2f} lakh")
print(f"99th percentile price: ₹{price_p99:.2f} lakh")

print("\nMarket concentration:")
print(
    f"Listings priced between ₹1Cr and ₹10Cr: "
    f"{price_segment_share.loc[['1Cr-2Cr', '2Cr-5Cr', '5Cr-10Cr']].sum():.2f}%"
)
print(
    f"Most common price band: "
    f"{price_segment_summary.idxmax()} "
    f"({price_segment_share.max():.2f}%)"
)

print("\nProperty mix:")
print(
    f"Most common property type: "
    f"{dominant_property_type} "
    f"({dominant_property_type_count:,} listings)"
)
print(
    f"Most common specified BHK group: "
    f"{dominant_bhk} BHK "
    f"({dominant_bhk_count:,} listings)"
)

print("\nLocality findings:")
print(
    f"Highest listing volume: "
    f"{top_locality} ({top_locality_count:,} listings)"
)
print(
    f"Highest median price among localities with at least 20 listings: "
    f"{highest_price_locality} "
    f"(₹{highest_price_locality_value:.2f} lakh)"
)
print(
    f"Highest median rate among localities with at least 20 listings: "
    f"{highest_rate_locality} "
    f"(₹{highest_rate_value:,.0f}/sqft)"
)

print("\nValue-position findings:")
print(
    f"Listings with a locality/property-type benchmark: "
    f"{benchmarked_count:,} ({benchmarked_count / total_listings * 100:.2f}%)"
)

for position in [
    "Lower-Priced",
    "Market-Aligned",
    "Premium-Priced"
]:
    count = value_position_summary.get(position, 0)
    share = count / benchmarked_count * 100
    print(f"{position}: {count:,} ({share:.2f}%)")


print("\nBHK comparison:")
print(bhk_summary.round(2).to_string())

print("\nRERA comparison:")
print(rera_summary.round(2).to_string())

print("\nReady-to-move vs Under Construction:")
print(status_summary.round(2).to_string())

PropertyLens - Final Market Findings

Market overview:
Total listings analysed: 14,225
Median property price: ₹262.00 lakh
Median property area: 2,015 sqft
Median rate: ₹12,379/sqft
90th percentile price: ₹766.00 lakh
99th percentile price: ₹2860.00 lakh

Market concentration:
Listings priced between ₹1Cr and ₹10Cr: 78.98%
Most common price band: 2Cr-5Cr (39.21%)

Property mix:
Most common property type: Apartment (10,895 listings)
Most common specified BHK group: 3 BHK (6,151 listings)

Locality findings:
Highest listing volume: Sector 65 (857 listings)
Highest median price among localities with at least 20 listings: Sector 42 (₹5050.00 lakh)
Highest median rate among localities with at least 20 listings: Sector 42 (₹56,698/sqft)

Value-position findings:
Listings with a locality/property-type benchmark: 12,635 (88.82%)
Lower-Priced: 4,182 (33.10%)
Market-Aligned: 4,729 (37.43%)
Premium-Priced: 3,724 (29.47%)

BHK comparison:
           listings  median_price_lakh  median_area_sqft  m

## Key Takeaways

- The dataset contains 14,225 property listings after duplicate removal and data cleaning.
- The median listed property price is ₹2.62 crore, while the higher mean reflects the strong right-skew in property prices.
- Listings between ₹1 crore and ₹10 crore account for 78.98% of the analysed dataset, with the ₹2–5 crore range being the largest segment.
- Apartments are the dominant property type, and 3 BHK is the most common specified configuration.
- Sector 65 has the highest listing volume, while Sector 42 has the highest median price and median rate among localities with at least 20 listings.
- Property size, BHK configuration and property type show meaningful relationships with listed price, although the strength of these relationships varies across the dataset.
- RERA-approved listings have a higher median rate per sqft than non-approved listings, but the groups have different property and status compositions, so this analysis does not establish a causal RERA price premium.
- Under-construction listings have higher median price and rate per sqft than ready-to-move listings, but the difference should not be interpreted as being caused by construction status alone.
- 88.82% of listings could be compared with a locality and property-type benchmark. Among these listings, 37.43% were market-aligned, 33.10% were lower-priced and 29.47% were premium-priced relative to their comparable benchmark.
- The analysis shows substantial variation in property pricing across localities, property types, sizes and configurations.